# Train & evaluate on Stanford Cars (Colab GPU)

**Before starting:** Runtime → Change runtime type → **T4 GPU**.

**Always run the cells top to bottom.** Free Colab wipes the machine when it disconnects. If that happens, just run from the top again: results are saved to your Google Drive (`MyDrive/item-identifier-runs`), and finished steps are skipped automatically, so only the dataset download is repeated.

Every number this notebook prints is a real measurement. Copy them into the README results table.

## 0. Setup (run every time you connect)

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')

%cd /content
if not os.path.isdir('/content/item-identifier'):
    !git clone -q https://github.com/manojrevenkar12-web/item-identifier.git
%cd /content/item-identifier
!git pull -q
!pip -q install -e ".[export,serve,dev]" datasets

# Results live on Google Drive so a disconnect never loses a finished training run.
RUNS = '/content/drive/MyDrive/item-identifier-runs'
os.makedirs(RUNS, exist_ok=True)
if not os.path.islink('runs'):
    !rm -rf runs
    os.symlink(RUNS, 'runs')

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('GPU:', gpu or 'NONE - switch the runtime to T4 GPU before training!')
!itemid --help > /dev/null && echo "itemid installed OK" 

In [ ]:
!pytest -q   # sanity check: should end with '22 passed'

## 1. Data (~10–15 min; repeated after every disconnect, that's normal)
Stanford Cars (196 models, 49 makes) plus corrupted test copies for the robustness evaluation.

In [ ]:
import os
if os.path.isdir('data/stanford_cars/test_contrast'):
    print('dataset already prepared')
else:
    !python scripts/prepare_stanford_cars.py --out data/stanford_cars --robustness

## 2. Fine-tune DINOv2 ViT-S/14 and the ResNet-50 baseline
Each cell skips itself if that model is already trained (saved on Drive).

In [ ]:
import os
if os.path.exists('runs/cars_dinov2_s/model.pt'):
    print('DINOv2 already trained - skipping')
else:
    !itemid train --config configs/stanford_cars.yaml

In [ ]:
import os
if os.path.exists('runs/cars_resnet50/model.pt'):
    print('ResNet-50 already trained - skipping')
else:
    !itemid train --config configs/baseline_resnet50.yaml

## 3. Held-out evaluation, robustness, export, latency, gates

In [ ]:
import os
for run, cfg in [("runs/cars_dinov2_s", "configs/stanford_cars.yaml"),
                 ("runs/cars_resnet50", "configs/baseline_resnet50.yaml")]:
    if not os.path.exists(f"{run}/model.pt"):
        print(f"{run}: not trained yet - run section 2 first")
        continue
    !itemid evaluate --config {cfg} --checkpoint {run}/model.pt --split test

In [ ]:
import json, os
run = "runs/cars_dinov2_s"
assert os.path.exists(f"{run}/report_test.json"), "Run section 2 and the evaluation cell above first."
corruptions = ["motion_blur", "jpeg_compression", "gaussian_noise", "pixelate", "contrast"]
for c in corruptions:
    if not os.path.exists(f"{run}/report_{c}.json"):
        !itemid evaluate --config configs/stanford_cars.yaml --checkpoint {run}/model.pt --split test_{c} --out {run}/report_{c}.json > /dev/null
print(f"{'split':20} top1   ECE    coverage  precision@accepted")
for name in ["test"] + corruptions:
    r = json.load(open(f"{run}/report_{name}.json"))
    p = r["selective"]["selective_precision"]
    print(f"{name:20} {r['top1']:.3f}  {r['calibration']['ece']:.3f}  {r['selective']['coverage']:.3f}     "
          f"{'-' if p is None else round(p, 3)}")

In [ ]:
import json
for run in ["runs/cars_dinov2_s", "runs/cars_resnet50"]:
    try:
        r = json.load(open(f"{run}/report_test.json"))
    except FileNotFoundError:
        continue
    c = r["calibration"]
    print(f"{run:22} top1={r['top1']:.4f} top5={r['top5']:.4f} ECE {c['ece_uncalibrated']:.3f} -> {c['ece']:.3f} "
          f"coverage={r['selective']['coverage']:.3f} precision={r['selective']['selective_precision']}")

In [ ]:
!itemid export --checkpoint runs/cars_dinov2_s/model.pt --out runs/cars_dinov2_s/model.onnx
!itemid bench --checkpoint runs/cars_dinov2_s/model.pt --onnx runs/cars_dinov2_s/model.onnx --threads 2 --out runs/cars_dinov2_s/latency.json
!itemid gate --config configs/stanford_cars.yaml --report runs/cars_dinov2_s/report_test.json --latency runs/cars_dinov2_s/latency.json

In [ ]:
!itemid prioritize --report runs/cars_dinov2_s/report_test.json | head -40

## 4. Active-learning experiment (optional, ~2–3 h; run it in a separate session)

In [ ]:
!itemid al-simulate --config configs/cars_active_learning.yaml --strategies random,margin,margin_diverse --rounds 3 --budget 700 --out runs/al_cars
from IPython.display import Image; Image('runs/al_cars/al_curve.png')

## 5. Results
Everything is already saved in Google Drive → `MyDrive/item-identifier-runs`. Open `cars_dinov2_s/report_test.html` there for the full visual report.